# Multilingual Neural Models for Turkic Languages (Glot500)

TurkicNLP includes multilingual neural models based on the **Glot500** backbone
that provide POS tagging, dependency parsing, morphological analysis, and
lemmatisation across many Turkic languages using a single shared model.

These models are particularly powerful because they support:

- **Trained languages** (10): Turkish, Azerbaijani, Uzbek, Turkmen, Kazakh,
  Kyrgyz, Bashkir, Tatar, Uyghur, Ottoman Turkish
- **Zero-shot languages** (via proxy embeddings): Karakalpak, Kumyk, Sakha

This notebook demonstrates three capabilities:

| Feature | Processor | Backend |
|---------|-----------|---------|
| POS tagging + Dependency parsing | `pos`, `depparse` | `multilingual_glot500` |
| Morphological analysis (UPOS + UD features + lemma) | `morph_neural` | multilingual Glot500 morph |
| Backend comparison | Stanza vs Glot500 | side-by-side |

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Download Models

In [ ]:
# Download for multiple languages — the Glot500 backbone is shared
for lang in ["tur", "kaz", "uzb", "kaa"]:
    turkicnlp.download(lang)

## 2. Multilingual POS Tagging and Dependency Parsing

The Glot500-based POS tagger and dependency parser use a shared model with per-language embeddings. Use `pos_backend="multilingual_glot500"` and `depparse_backend="multilingual_glot500"` to select this backend.

In [ ]:
# Turkish
nlp_tur = Pipeline(
    "tur",
    processors=["tokenize", "pos", "depparse"],
    pos_backend="multilingual_glot500",
    depparse_backend="multilingual_glot500",
)
doc = nlp_tur("Ahmet bugün okula gitti.")
print("=== Turkish ===")
print(f"{'Word':<15} {'UPOS':<8} {'Head':<5} {'Deprel'}")
print("-" * 40)
for w in doc.words:
    print(f"{w.text:<15} {w.upos:<8} {w.head!s:<5} {w.deprel}")

In [ ]:
# Kazakh (Cyrillic)
nlp_kaz = Pipeline(
    "kaz",
    processors=["tokenize", "pos", "depparse"],
    pos_backend="multilingual_glot500",
    depparse_backend="multilingual_glot500",
    script="Cyrl",
)
doc = nlp_kaz("Ахмет бүгін мектепке барды.")
print("=== Kazakh ===")
print(f"{'Word':<15} {'UPOS':<8} {'Head':<5} {'Deprel'}")
print("-" * 40)
for w in doc.words:
    print(f"{w.text:<15} {w.upos:<8} {w.head!s:<5} {w.deprel}")

In [ ]:
# Uzbek (Latin)
nlp_uzb = Pipeline(
    "uzb",
    processors=["tokenize", "pos", "depparse"],
    pos_backend="multilingual_glot500",
    depparse_backend="multilingual_glot500",
)
doc = nlp_uzb("Ahmat bugun maktabga ketdi.")
print("=== Uzbek ===")
print(f"{'Word':<15} {'UPOS':<8} {'Head':<5} {'Deprel'}")
print("-" * 40)
for w in doc.words:
    print(f"{w.text:<15} {w.upos:<8} {w.head!s:<5} {w.deprel}")

## 3. Zero-shot Parsing for Unseen Languages

The multilingual model can parse languages it was never directly trained on, using proxy embeddings from related languages. Karakalpak (Kipchak, close to Uzbek) is one such zero-shot language.

In [ ]:
# Karakalpak — zero-shot via Uzbek proxy embedding
nlp_kaa = Pipeline(
    "kaa",
    processors=["tokenize", "pos", "depparse"],
    pos_backend="multilingual_glot500",
    depparse_backend="multilingual_glot500",
)
doc = nlp_kaa("Qiz dostina xat jazdi.")
print("=== Karakalpak (zero-shot) ===")
print(f"{'Word':<15} {'UPOS':<8} {'Head':<5} {'Deprel'}")
print("-" * 40)
for w in doc.words:
    print(f"{w.text:<15} {w.upos:<8} {w.head!s:<5} {w.deprel}")
print("\nCoNLL-U:\n", doc.to_conllu())

## 4. Neural Morphological Analysis (Glot500 Morph)

The `morph_neural` processor provides UPOS tags, UD morphological features, and lemmatisation for 21 Turkic languages using the Glot500 morph model. This is broader than the Stanza-based models.

In [ ]:
turkicnlp.download("tur", processors=["tokenize", "morph_neural"])

nlp_morph = Pipeline(
    "tur",
    processors=["tokenize", "morph_neural"],
)
doc = nlp_morph("Çocuklar okula gidiyorlar.")
print("=== Turkish — Neural Morphology ===")
print(f"{'Word':<20} {'UPOS':<8} {'Lemma':<15} {'Features'}")
print("-" * 70)
for w in doc.words:
    print(f"{w.text:<20} {w.upos:<8} {w.lemma:<15} {w.feats}")

In [ ]:
# Neural morph for low-resource languages
for lang, text, label in [
    ("sah", "Мин оскуолаҕа бардым.", "Sakha"),
    ("kaa", "Men mektepke bardim.", "Karakalpak (zero-shot)"),
]:
    turkicnlp.download(lang, processors=["tokenize", "morph_neural"])
    nlp = Pipeline(lang, processors=["tokenize", "morph_neural"])
    doc = nlp(text)
    print(f"\n=== {label} ===")
    for w in doc.words:
        print(f"  {w.text:<20} upos={w.upos:<8} lemma={w.lemma:<15} feats={w.feats}")

## 5. Backend Comparison: Stanza vs Glot500

For languages with both Stanza and Glot500 models (e.g., Turkish, Kazakh), you can compare outputs side-by-side. Stanza models are language-specific and typically more accurate for high-resource languages, while Glot500 provides broader coverage.

In [ ]:
text = "Ahmet bugün okula gitti."

# Stanza backend (language-specific, trained on Turkish IMST treebank)
nlp_stanza = Pipeline(
    "tur",
    processors=["tokenize", "pos", "lemma", "depparse"],
)
doc_stanza = nlp_stanza(text)

# Glot500 backend (multilingual)
nlp_glot = Pipeline(
    "tur",
    processors=["tokenize", "pos", "depparse"],
    pos_backend="multilingual_glot500",
    depparse_backend="multilingual_glot500",
)
doc_glot = nlp_glot(text)

print(f"{'Word':<15} {'Stanza UPOS':<13} {'Glot500 UPOS':<14} {'Stanza Dep':<12} {'Glot500 Dep'}")
print("-" * 70)
# Both backends use the same rule-based tokenizer, so word counts match
for ws, wg in zip(doc_stanza.words, doc_glot.words):
    match_pos = "✓" if ws.upos == wg.upos else "✗"
    match_dep = "✓" if ws.deprel == wg.deprel else "✗"
    print(f"{ws.text:<15} {ws.upos:<13} {wg.upos:<14} {ws.deprel:<12} {wg.deprel} {match_pos}{match_dep}")

## 6. Processing Multiple Languages in a Loop

The multilingual backend makes it easy to process text from many Turkic languages in a uniform way.

In [ ]:
sentences = [
    ("tur", "Bugün hava güzel.", "Turkish"),
    ("kaz", "Бүгін ауа райы жақсы.", "Kazakh"),
    ("uzb", "Bugun ob-havo yaxshi.", "Uzbek"),
    ("tat", "Бүген һава матур.", "Tatar"),
    ("kir", "Бүгүн аба ырайы жакшы.", "Kyrgyz"),
]

for lang, text, label in sentences:
    nlp = Pipeline(
        lang,
        processors=["tokenize", "pos", "depparse"],
        pos_backend="multilingual_glot500",
        depparse_backend="multilingual_glot500",
    )
    doc = nlp(text)
    tags = " ".join(f"{w.text}/{w.upos}" for w in doc.words)
    print(f"[{label:<10}] {tags}")